In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import pickle
import warnings
warnings.filterwarnings('ignore')

import implicit
from implicit.als import AlternatingLeastSquares

history = pd.read_parquet('artifacts/history_interactions.parquet')
item_feat_mh = pd.read_parquet('artifacts/item_features_multihot.parquet')

print(f"History: {history.shape}")

# ============================================================
# 1. ALS 128 FACTORS
# ============================================================
user_ids = history['steamid'].unique()
all_candidate_items = np.union1d(
    pd.read_parquet('artifacts/train_reranker_base.parquet')['appid'].unique(),
    pd.read_parquet('artifacts/test_reranker_base.parquet')['appid'].unique()
)
all_items = np.union1d(history['appid'].unique(), all_candidate_items)

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {a: i for i, a in enumerate(all_items)}
idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: a for a, i in item2idx.items()}

n_users, n_items = len(user2idx), len(item2idx)
print(f"Matrix: {n_users} × {n_items}")

rows = history['steamid'].map(user2idx).values
cols = history['appid'].map(item2idx).values
confidence = (history['target'].values * 2.0 + np.log1p(history['playtime_forever'].values) * 0.5).astype(np.float32)
confidence = np.clip(confidence, 1.0, None)

user_item_matrix = csr_matrix((confidence, (rows, cols)), shape=(n_users, n_items), dtype=np.float32)

N_FACTORS = 128
print(f"\nTraining ALS with {N_FACTORS} factors...")
als_model = AlternatingLeastSquares(
    factors=N_FACTORS,
    regularization=0.01,
    iterations=30,
    random_state=42,
    use_gpu=False,
)
als_model.fit(user_item_matrix)
print("Done!")

user_factors = als_model.user_factors  # (n_users, 128)
item_factors = als_model.item_factors  # (n_items, 128)
print(f"User factors: {user_factors.shape}, Item factors: {item_factors.shape}")

# Normalized
from numpy.linalg import norm
item_norms = norm(item_factors, axis=1, keepdims=True).clip(1e-8)
item_factors_normed = item_factors / item_norms
user_norms = norm(user_factors, axis=1, keepdims=True).clip(1e-8)
user_factors_normed = user_factors / user_norms

# ============================================================
# 2. PRECOMPUTE USER CENTROIDS + DIVERSE TOP-K
# ============================================================
print("\nComputing user centroids and similarity profiles...")

binary_matrix = (user_item_matrix > 0).astype(np.float32)

user_centroids = np.zeros((n_users, N_FACTORS), dtype=np.float32)
user_top_by_conf = {}     # top-10 by confidence (=engagement)
user_taste_std = np.zeros(n_users, dtype=np.float32)  # taste diversity

for uid in range(n_users):
    played_idx = binary_matrix[uid].nonzero()[1]
    if len(played_idx) == 0:
        continue
    
    weights = user_item_matrix[uid, played_idx].toarray().flatten()
    weights_norm = weights / weights.sum()
    
    centroid = (item_factors_normed[played_idx] * weights_norm[:, None]).sum(axis=0)
    c_norm = norm(centroid)
    if c_norm > 1e-8:
        centroid = centroid / c_norm
    user_centroids[uid] = centroid
    
    # Top-10 by confidence
    top_k = min(10, len(played_idx))
    top_indices = played_idx[np.argsort(-weights)][:top_k]
    user_top_by_conf[idx2user[uid]] = top_indices
    
    # Taste diversity: std of item vectors in history
    if len(played_idx) > 1:
        hist_vecs = item_factors_normed[played_idx]
        user_taste_std[uid] = hist_vecs.std(axis=0).mean()

# User centroid DataFrame
user_centroid_df = pd.DataFrame({
    'steamid': [idx2user[i] for i in range(n_users)],
    'user_taste_diversity': user_taste_std
})

print(f"Centroids computed for {(user_centroids.sum(axis=1) != 0).sum()} users")

# ============================================================
# 3. SAVE ALL
# ============================================================
# Factor DataFrames
uf_cols = [f'uf_{i}' for i in range(N_FACTORS)]
if_cols = [f'if_{i}' for i in range(N_FACTORS)]

uf_df = pd.DataFrame(user_factors, columns=uf_cols)
uf_df['steamid'] = [idx2user[i] for i in range(n_users)]

if_df = pd.DataFrame(item_factors, columns=if_cols)
if_df['appid'] = [idx2item[i] for i in range(n_items)]

uf_df.to_parquet('artifacts/v4_user_factors.parquet', index=False)
if_df.to_parquet('artifacts/v4_item_factors.parquet', index=False)
np.save('artifacts/v4_item_factors_normed.npy', item_factors_normed)
np.save('artifacts/v4_user_centroids.npy', user_centroids)
user_centroid_df.to_parquet('artifacts/v4_user_centroid_meta.parquet', index=False)

with open('artifacts/v4_user_top_conf.pkl', 'wb') as f:
    pickle.dump(user_top_by_conf, f)
with open('artifacts/v4_user2idx.pkl', 'wb') as f:
    pickle.dump(user2idx, f)
with open('artifacts/v4_item2idx.pkl', 'wb') as f:
    pickle.dump(item2idx, f)

print("\nAll v4 ALS artifacts saved!")
print(f"Factors: {N_FACTORS}")


History: (342880, 10)
Matrix: 4564 × 19268

Training ALS with 128 factors...


  0%|          | 0/30 [00:00<?, ?it/s]

Done!
User factors: (4564, 128), Item factors: (19268, 128)

Computing user centroids and similarity profiles...
Centroids computed for 4564 users

All v4 ALS artifacts saved!
Factors: 128


In [2]:
import pandas as pd
import numpy as np
import ast
import pickle
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# LOAD EVERYTHING
# ============================================================
history = pd.read_parquet('artifacts/history_interactions.parquet')
games = pd.read_csv('data/game_details.csv')
users = pd.read_csv('data/unique_users.csv')
item_feat_mh = pd.read_parquet('artifacts/item_features_multihot.parquet')

# v2 artifacts
item_pop = pd.read_parquet('artifacts/v2_item_pop.parquet')
item_dev_agg = pd.read_parquet('artifacts/v2_item_dev.parquet')
item_pub_agg = pd.read_parquet('artifacts/v2_item_pub.parquet')
user_stats = pd.read_parquet('artifacts/v2_user_stats.parquet')
user_plat = pd.read_parquet('artifacts/v2_user_plat.parquet')
user_free = pd.read_parquet('artifacts/v2_user_free.parquet')
user_dev = pd.read_parquet('artifacts/v2_user_dev.parquet')
user_pub = pd.read_parquet('artifacts/v2_user_pub.parquet')
dev_per_appid = pd.read_parquet('artifacts/v2_dev_per_appid.parquet')
pub_per_appid = pd.read_parquet('artifacts/v2_pub_per_appid.parquet')

# v3 target encodings
item_dev_te = pd.read_parquet('artifacts/v3_item_dev_te.parquet')
item_pub_te = pd.read_parquet('artifacts/v3_item_pub_te.parquet')
item_genre_te = pd.read_parquet('artifacts/v3_item_genre_te.parquet')
user_dev_te = pd.read_parquet('artifacts/v3_user_dev_te.parquet')
dev_exploded = pd.read_parquet('artifacts/v3_dev_exploded.parquet')

# v4 ALS
uf_df = pd.read_parquet('artifacts/v4_user_factors.parquet')
if_df = pd.read_parquet('artifacts/v4_item_factors.parquet')
item_factors_normed = np.load('artifacts/v4_item_factors_normed.npy')
user_centroids = np.load('artifacts/v4_user_centroids.npy')
user_centroid_meta = pd.read_parquet('artifacts/v4_user_centroid_meta.parquet')

with open('artifacts/v4_user_top_conf.pkl', 'rb') as f:
    user_top_conf = pickle.load(f)
with open('artifacts/v4_user2idx.pkl', 'rb') as f:
    user2idx = pickle.load(f)
with open('artifacts/v4_item2idx.pkl', 'rb') as f:
    item2idx = pickle.load(f)

N_FACTORS = 128
uf_cols = [f'uf_{i}' for i in range(N_FACTORS)]
if_cols = [f'if_{i}' for i in range(N_FACTORS)]

mh_genre_cols = [c for c in item_feat_mh.columns if c.startswith('genres_')]
mh_cat_cols = [c for c in item_feat_mh.columns if c.startswith('categories_')]
mh_cols = mh_genre_cols + mh_cat_cols

# User profile
user_profile = users[['steamid', 'loccountrycode', 'timecreated']].copy()
user_profile['loccountrycode'] = user_profile['loccountrycode'].fillna('UNKNOWN')
max_time = 1776010854
user_profile['account_age_days'] = ((max_time - user_profile['timecreated'].fillna(max_time)) / 86400).clip(lower=0)
country_counts = user_profile['loccountrycode'].value_counts()
user_profile['country_freq'] = user_profile['loccountrycode'].map(country_counts).fillna(0).astype(int)
user_profile = user_profile.drop(columns=['timecreated'])

# Game type
game_type = games[['appid', 'type']].copy()
game_type['type'] = game_type['type'].fillna('unknown')

print("All artifacts loaded!")

# ============================================================
# PRE-COMPUTE: Genre/Category affinity (playtime-weighted)
# ============================================================
print("\nComputing playtime-weighted affinities...")

hist_mh = history[['steamid', 'appid', 'playtime_forever']].merge(
    item_feat_mh[['appid'] + mh_cols], on='appid', how='inner'
)
hist_mh['log_pt'] = np.log1p(hist_mh['playtime_forever']).astype('float32')

# Count-based
user_game_counts = hist_mh.groupby('steamid').size().reset_index(name='_cnt')
user_aff_count = hist_mh.groupby('steamid')[mh_cols].sum().reset_index()
user_aff_count = user_aff_count.merge(user_game_counts, on='steamid')
for col in mh_cols:
    user_aff_count[f'uaff_{col}'] = (user_aff_count[col] / user_aff_count['_cnt']).astype('float32')
uaff_count_cols = [f'uaff_{c}' for c in mh_cols]
user_aff_count = user_aff_count[['steamid'] + uaff_count_cols]

# Playtime-weighted
for col in mh_cols:
    hist_mh[f'w_{col}'] = hist_mh[col].values * hist_mh['log_pt'].values
w_cols = [f'w_{c}' for c in mh_cols]
user_aff_wt = hist_mh.groupby('steamid')[w_cols].sum().reset_index()
user_total_wt = hist_mh.groupby('steamid')['log_pt'].sum().reset_index().rename(columns={'log_pt': '_tw'})
user_aff_wt = user_aff_wt.merge(user_total_wt, on='steamid')
for col in mh_cols:
    user_aff_wt[f'uwaff_{col}'] = (user_aff_wt[f'w_{col}'] / user_aff_wt['_tw'].clip(lower=0.001)).astype('float32')
uwaff_cols = [f'uwaff_{c}' for c in mh_cols]
user_aff_wt = user_aff_wt[['steamid'] + uwaff_cols]

print(f"Affinity computed for {len(user_aff_count)} users")


# ============================================================
# ASSEMBLY FUNCTION
# ============================================================
def assemble_v4(base_path):
    print(f"\n{'='*60}")
    print(f"Assembling v4: {base_path}")
    print('='*60)
    
    df = pd.read_parquet(base_path)
    df = df.rename(columns={'score': 'als_score', 'rank': 'als_rank'})
    N = len(df)
    
    # ============================================================
    # 1. ALS SCORE FEATURES
    # ============================================================
    print("  [1] ALS score features...")
    df['als_score_log'] = np.log1p(df['als_score'])
    df['als_rank_inv'] = 1.0 / df['als_rank']
    df['als_rank_inv_sqrt'] = 1.0 / np.sqrt(df['als_rank'])
    df['als_rank_norm'] = df['als_rank'] / 300.0
    
    als_g = df.groupby('steamid')['als_score'].agg(['mean', 'std', 'max', 'min']).reset_index()
    als_g.columns = ['steamid', '_am', '_as', '_ax', '_an']
    df = df.merge(als_g, on='steamid')
    df['als_score_zscore'] = (df['als_score'] - df['_am']) / df['_as'].clip(lower=0.001)
    df['als_score_minmax'] = (df['als_score'] - df['_an']) / (df['_ax'] - df['_an']).clip(lower=0.001)
    df.drop(columns=['_am', '_as', '_ax', '_an'], inplace=True)
    
    # ============================================================
    # 2. ALL 128 ALS FACTOR PRODUCTS (КЛЮЧЕВОЕ!)
    # ============================================================
    print("  [2] ALS 128 factor cross-products...")
    df = df.merge(uf_df, on='steamid', how='left')
    df = df.merge(if_df, on='appid', how='left')
    for c in uf_cols + if_cols:
        df[c] = df[c].fillna(0)
    
    uf_vals = df[uf_cols].values.astype(np.float32)
    if_vals = df[if_cols].values.astype(np.float32)
    
    # Dot product
    df['als_dot_v4'] = np.sum(uf_vals * if_vals, axis=1)
    
    # ALL element-wise products
    uf_if_product = uf_vals * if_vals
    for i in range(N_FACTORS):
        df[f'uf_x_if_{i}'] = uf_if_product[:, i]
    
    # Norms
    df['uf_norm'] = np.linalg.norm(uf_vals, axis=1)
    df['if_norm'] = np.linalg.norm(if_vals, axis=1)
    df['als_cosine_v4'] = df['als_dot_v4'] / (df['uf_norm'] * df['if_norm']).clip(lower=1e-8)
    
    # Drop raw factors
    df.drop(columns=uf_cols + if_cols, inplace=True)
    
    # ============================================================
    # 3. VECTORIZED CO-OCCURRENCE SIMILARITY
    # ============================================================
    print("  [3] Vectorized item-item similarity...")
    
    steamids = df['steamid'].values
    appids = df['appid'].values
    
    # Map to indices
    user_idx_arr = np.array([user2idx.get(s, -1) for s in steamids], dtype=np.int32)
    item_idx_arr = np.array([item2idx.get(a, -1) for a in appids], dtype=np.int32)
    
    # Centroid similarity (vectorized)
    valid_mask = (user_idx_arr >= 0) & (item_idx_arr >= 0)
    centroid_sim = np.zeros(N, dtype=np.float32)
    
    if valid_mask.any():
        u_centroids = user_centroids[user_idx_arr[valid_mask]]  # (n_valid, 128)
        i_vecs = item_factors_normed[item_idx_arr[valid_mask]]   # (n_valid, 128)
        centroid_sim[valid_mask] = np.sum(u_centroids * i_vecs, axis=1)
    
    df['centroid_sim'] = centroid_sim
    
    # Top-K similarity (batch per user)
    print("    Computing top-K similarity (batched)...")
    max_sim_arr = np.zeros(N, dtype=np.float32)
    avg_sim_arr = np.zeros(N, dtype=np.float32)
    min_sim_arr = np.zeros(N, dtype=np.float32)
    
    df['_row_idx'] = np.arange(N)
    grouped = df.groupby('steamid')['_row_idx'].apply(list).reset_index()
    
    for _, row in grouped.iterrows():
        sid = row['steamid']
        indices = row['_row_idx']
        
        if sid not in user_top_conf:
            continue
        
        top_item_idx = user_top_conf[sid]
        if len(top_item_idx) == 0:
            continue
        
        # Candidate item indices for this user
        cand_appids = appids[indices]
        cand_item_idx = np.array([item2idx.get(a, -1) for a in cand_appids])
        valid = cand_item_idx >= 0
        
        if not valid.any():
            continue
        
        # Batch similarity: (n_top, 128) @ (128, n_candidates) → (n_top, n_candidates)
        top_vecs = item_factors_normed[top_item_idx]  # (n_top, 128)
        cand_vecs = item_factors_normed[cand_item_idx[valid]]  # (n_valid_cand, 128)
        sim_matrix = top_vecs @ cand_vecs.T  # (n_top, n_valid_cand)
        
        valid_indices = np.array(indices)[valid]
        max_sim_arr[valid_indices] = sim_matrix.max(axis=0)
        avg_sim_arr[valid_indices] = sim_matrix.mean(axis=0)
        min_sim_arr[valid_indices] = sim_matrix.min(axis=0)
    
    df['max_sim_top10'] = max_sim_arr
    df['avg_sim_top10'] = avg_sim_arr
    df['min_sim_top10'] = min_sim_arr
    df['sim_range_top10'] = max_sim_arr - min_sim_arr
    df.drop(columns=['_row_idx'], inplace=True)
    
    # ============================================================
    # 4. USER FEATURES
    # ============================================================
    print("  [4] User features...")
    df = df.merge(user_profile, on='steamid', how='left')
    df = df.merge(user_stats, on='steamid', how='left')
    df = df.merge(user_plat, on='steamid', how='left')
    df = df.merge(user_free, on='steamid', how='left')
    df = df.merge(user_centroid_meta, on='steamid', how='left')
    
    # ============================================================
    # 5. ITEM FEATURES
    # ============================================================
    print("  [5] Item features...")
    item_compact = item_feat_mh[['appid', 'is_free', 'recommendations_log', 'age_years', 'platforms_count']].copy()
    df = df.merge(item_compact, on='appid', how='left')
    df = df.merge(item_pop, on='appid', how='left')
    df = df.merge(item_dev_agg, on='appid', how='left')
    df = df.merge(item_pub_agg, on='appid', how='left')
    df = df.merge(item_dev_te, on='appid', how='left')
    df = df.merge(item_pub_te, on='appid', how='left')
    df = df.merge(item_genre_te, on='appid', how='left')
    df = df.merge(game_type, on='appid', how='left')
    df['type'] = df['type'].fillna('unknown')
    
    # ============================================================
    # 6. GENRE/CATEGORY MATCH SCORES (были в v2, пропущены в v3!)
    # ============================================================
    print("  [6] Genre/Category match scores...")
    df = df.merge(user_aff_count, on='steamid', how='left')
    df = df.merge(user_aff_wt, on='steamid', how='left')
    
    # Подгрузим item multi-hot для match
    item_mh_full = item_feat_mh[['appid'] + mh_cols]
    df = df.merge(item_mh_full, on='appid', how='left', suffixes=('', '_item'))
    
    # Genre match (count-based + weighted)
    genre_dot = np.zeros(N, dtype='float32')
    genre_wdot = np.zeros(N, dtype='float32')
    u_gnorm_sq = np.zeros(N, dtype='float32')
    i_gnorm_sq = np.zeros(N, dtype='float32')
    uw_gnorm_sq = np.zeros(N, dtype='float32')
    
    cat_dot = np.zeros(N, dtype='float32')
    cat_wdot = np.zeros(N, dtype='float32')
    u_cnorm_sq = np.zeros(N, dtype='float32')
    i_cnorm_sq = np.zeros(N, dtype='float32')
    
    for col in mh_genre_cols:
        i_val = df[col].fillna(0).values.astype('float32')
        u_val = df[f'uaff_{col}'].fillna(0).values.astype('float32')
        uw_val = df[f'uwaff_{col}'].fillna(0).values.astype('float32')
        
        genre_dot += u_val * i_val
        genre_wdot += uw_val * i_val
        u_gnorm_sq += u_val ** 2
        i_gnorm_sq += i_val ** 2
        uw_gnorm_sq += uw_val ** 2
    
    for col in mh_cat_cols:
        i_val = df[col].fillna(0).values.astype('float32')
        u_val = df[f'uaff_{col}'].fillna(0).values.astype('float32')
        uw_val = df[f'uwaff_{col}'].fillna(0).values.astype('float32')
        
        cat_dot += u_val * i_val
        cat_wdot += uw_val * i_val
        u_cnorm_sq += u_val ** 2
        i_cnorm_sq += i_val ** 2
    
    denom_g = np.sqrt(u_gnorm_sq * i_gnorm_sq).clip(1e-8)
    denom_gw = np.sqrt(uw_gnorm_sq * i_gnorm_sq).clip(1e-8)
    denom_c = np.sqrt(u_cnorm_sq * i_cnorm_sq).clip(1e-8)
    
    df['genre_match_dot'] = genre_dot
    df['genre_match_cosine'] = genre_dot / denom_g
    df['genre_wmatch_dot'] = genre_wdot
    df['genre_wmatch_cosine'] = genre_wdot / denom_gw
    df['cat_match_dot'] = cat_dot
    df['cat_match_cosine'] = cat_dot / denom_c
    df['cat_wmatch_dot'] = cat_wdot
    df['total_match'] = genre_dot + cat_dot
    df['total_wmatch'] = genre_wdot + cat_wdot
    df['total_match_cosine'] = (df['genre_match_cosine'] + df['cat_match_cosine']) / 2
    
    # Drop affinity columns (match scores are enough)
    drop_aff = [c for c in df.columns if c.startswith('uaff_') or c.startswith('uwaff_')]
    # Also drop raw multi-hot from items (match scores capture the info)
    drop_mh = mh_cols
    df.drop(columns=drop_aff + drop_mh, inplace=True, errors='ignore')
    
    # ============================================================
    # 7. DEV/PUB CROSS-FEATURES
    # ============================================================
    print("  [7] Dev/Pub cross features...")
    
    # Developer affinity
    df_d = df[['steamid', 'appid']].merge(dev_per_appid, on='appid', how='left')
    df_d['developer'] = df_d['developer'].apply(lambda x: x if isinstance(x, list) else [])
    df_d = df_d.explode('developer')
    df_d = df_d.merge(user_dev, on=['steamid', 'developer'], how='left')
    df_d['user_dev_n_games'] = df_d['user_dev_n_games'].fillna(0)
    df_d['user_dev_total_pt'] = df_d['user_dev_total_pt'].fillna(0)
    
    da = df_d.groupby(['steamid', 'appid']).agg(
        user_dev_max_games=('user_dev_n_games', 'max'),
        user_dev_sum_games=('user_dev_n_games', 'sum'),
        user_dev_max_pt=('user_dev_total_pt', 'max'),
        user_dev_has_history=('user_dev_n_games', lambda x: int((x > 0).any())),
    ).reset_index()
    df = df.merge(da, on=['steamid', 'appid'], how='left')
    df['user_dev_max_pt_log'] = np.log1p(df['user_dev_max_pt'].fillna(0))
    
    # Publisher affinity
    df_p = df[['steamid', 'appid']].merge(pub_per_appid, on='appid', how='left')
    df_p['publisher'] = df_p['publisher'].apply(lambda x: x if isinstance(x, list) else [])
    df_p = df_p.explode('publisher')
    df_p = df_p.merge(user_pub, on=['steamid', 'publisher'], how='left')
    df_p['user_pub_n_games'] = df_p['user_pub_n_games'].fillna(0)
    
    pa = df_p.groupby(['steamid', 'appid']).agg(
        user_pub_max_games=('user_pub_n_games', 'max'),
        user_pub_has_history=('user_pub_n_games', lambda x: int((x > 0).any())),
    ).reset_index()
    df = df.merge(pa, on=['steamid', 'appid'], how='left')
    
    # User-Dev TE
    df_dt = df[['steamid', 'appid']].merge(dev_exploded, on='appid', how='left')
    df_dt = df_dt.merge(user_dev_te[['steamid', 'developer', 'user_dev_te_smooth']], 
                         on=['steamid', 'developer'], how='left')
    dt_agg = df_dt.groupby(['steamid', 'appid']).agg(
        user_dev_te_max=('user_dev_te_smooth', 'max'),
        user_dev_te_mean=('user_dev_te_smooth', 'mean'),
    ).reset_index()
    df = df.merge(dt_agg, on=['steamid', 'appid'], how='left')
    
    # ============================================================
    # 8. INTERACTION FEATURES
    # ============================================================
    print("  [8] Interaction features...")
    df['free_match'] = df['user_free_ratio'].fillna(0.5) * df['is_free'].fillna(0)
    df['als_x_centroid'] = df['als_score'] * df['centroid_sim']
    df['als_x_max_sim'] = df['als_score'] * df['max_sim_top10']
    df['als_x_dev_hist'] = df['als_score'] * df['user_dev_has_history'].fillna(0)
    df['als_dot_x_centroid'] = df['als_dot_v4'] * df['centroid_sim']
    df['als_dot_x_max_sim'] = df['als_dot_v4'] * df['max_sim_top10']
    df['als_x_genre_cos'] = df['als_score'] * df['genre_match_cosine']
    df['als_x_total_match'] = df['als_score'] * df['total_match_cosine']
    df['dev_te_x_als'] = df['item_dev_te_max'].fillna(0) * df['als_score']
    df['user_eng_x_item'] = df['user_avg_target_hist'].fillna(3) * df['item_avg_target_hist'].fillna(3)
    df['pop_vs_lib'] = df['item_n_players'].fillna(0) / df['user_total_games'].clip(lower=1)
    
    # Centroid × match combos
    df['centroid_x_genre'] = df['centroid_sim'] * df['genre_match_cosine']
    df['centroid_x_dev'] = df['centroid_sim'] * df['user_dev_has_history'].fillna(0)
    
    # User quality × item quality
    df['user_high_x_item_high'] = df['user_high_target_ratio'].fillna(0) * df['item_pct_high_target'].fillna(0)
    
    # ============================================================
    # 9. CLEANUP
    # ============================================================
    print("  [9] Cleanup...")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)
    for c in ['loccountrycode', 'type']:
        if c in df.columns:
            df[c] = df[c].fillna('unknown')
    
    assert len(df) == N, f"Row count changed: {N} -> {len(df)}"
    print(f"  Final: {df.shape}")
    return df


# ============================================================
# BUILD
# ============================================================
train_v4 = assemble_v4('artifacts/train_reranker_base.parquet')
test_v4 = assemble_v4('artifacts/test_reranker_base.parquet')

train_v4.sort_values('steamid', inplace=True)
test_v4.sort_values('steamid', inplace=True)

train_v4.to_parquet('artifacts/train_v4.parquet', index=False)
test_v4.to_parquet('artifacts/test_v4.parquet', index=False)

print(f"\nTrain: {train_v4.shape}, Test: {test_v4.shape}")
print(f"\nColumns ({len(train_v4.columns)}):")
for i, c in enumerate(train_v4.columns):
    print(f"  {i:3d}. {c} — {train_v4[c].dtype}")


All artifacts loaded!

Computing playtime-weighted affinities...
Affinity computed for 4564 users

Assembling v4: artifacts/train_reranker_base.parquet
  [1] ALS score features...
  [2] ALS 128 factor cross-products...
  [3] Vectorized item-item similarity...
    Computing top-K similarity (batched)...
  [4] User features...
  [5] Item features...
  [6] Genre/Category match scores...
  [7] Dev/Pub cross features...
  [8] Interaction features...
  [9] Cleanup...
  Final: (1064400, 232)

Assembling v4: artifacts/test_reranker_base.parquet
  [1] ALS score features...
  [2] ALS 128 factor cross-products...
  [3] Vectorized item-item similarity...
    Computing top-K similarity (batched)...
  [4] User features...
  [5] Item features...
  [6] Genre/Category match scores...
  [7] Dev/Pub cross features...
  [8] Interaction features...
  [9] Cleanup...
  Final: (266100, 232)

Train: (1064400, 232), Test: (266100, 232)

Columns (232):
    0. steamid — int64
    1. appid — int64
    2. als_score

In [3]:
import pandas as pd
import numpy as np
from catboost import CatBoostRanker, Pool
from catboost.utils import get_gpu_device_count
import mlflow

print(f"GPU: {get_gpu_device_count()}")

train = pd.read_parquet("artifacts/train_v4.parquet").sort_values("steamid")
test = pd.read_parquet("artifacts/test_v4.parquet").sort_values("steamid")

drop_cols = ["steamid", "appid", "target"]
cat_features = ["loccountrycode", "type"]

for cf in cat_features:
    train[cf] = train[cf].fillna("unknown").astype(str)
    test[cf] = test[cf].fillna("unknown").astype(str)

feature_cols = [c for c in train.columns if c not in drop_cols]
print(f"Features: {len(feature_cols)}")

X_train, y_train, q_train = train[feature_cols], train["target"], train["steamid"]
X_test, y_test, q_test = test[feature_cols], test["target"], test["steamid"]

train_pool = Pool(data=X_train, label=y_train, group_id=q_train, cat_features=cat_features)
test_pool = Pool(data=X_test, label=y_test, group_id=q_test, cat_features=cat_features)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Steam_RecSys_v4")

# ============================================================
# Run 1: PairLogitPairwise (наш лучший loss до сих пор)
# ============================================================
print("\n" + "="*60)
print("Run 1: PairLogitPairwise")
print("="*60)

with mlflow.start_run(run_name="v4_PairLogitPW"):
    params1 = {
        "iterations": 5000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "loss_function": "PairLogitPairwise",
        "custom_metric": ["NDCG:top=10", "MAP:top=10"],
        "eval_metric": "NDCG:top=10",
        "early_stopping_rounds": 200,
        "random_seed": 42,
        "task_type": "GPU",
        "devices": "0",
        "border_count": 128,
        "bootstrap_type": "Bayesian",
        "verbose": 200,
    }
    mlflow.log_params(params1)
    
    m1 = CatBoostRanker(**params1)
    m1.fit(train_pool, eval_set=test_pool)
    
    s1 = m1.get_best_score()["validation"]["NDCG:top=10;type=Base"]
    print(f"PairLogitPW NDCG@10: {s1:.4f} (iter {m1.get_best_iteration()})")
    mlflow.log_metric("best_ndcg_10", s1)
    m1.save_model("artifacts/catboost_v4_plp.cbm")

# ============================================================
# Run 2: YetiRank
# ============================================================
print("\n" + "="*60)
print("Run 2: YetiRank")
print("="*60)

with mlflow.start_run(run_name="v4_YetiRank"):
    params2 = {
        "iterations": 5000,
        "learning_rate": 0.03,
        "depth": 7,
        "l2_leaf_reg": 3.0,
        "loss_function": "YetiRank",
        "custom_metric": ["NDCG:top=10", "MAP:top=10"],
        "eval_metric": "NDCG:top=10",
        "early_stopping_rounds": 200,
        "random_seed": 42,
        "task_type": "GPU",
        "devices": "0",
        "verbose": 200,
    }
    mlflow.log_params(params2)
    
    m2 = CatBoostRanker(**params2)
    m2.fit(train_pool, eval_set=test_pool)
    
    s2 = m2.get_best_score()["validation"]["NDCG:top=10;type=Base"]
    print(f"YetiRank NDCG@10: {s2:.4f} (iter {m2.get_best_iteration()})")
    mlflow.log_metric("best_ndcg_10", s2)
    m2.save_model("artifacts/catboost_v4_yr.cbm")

# ============================================================
# Run 3: YetiRankPairwise
# ============================================================
print("\n" + "="*60)
print("Run 3: YetiRankPairwise")
print("="*60)

with mlflow.start_run(run_name="v4_YetiRankPairwise"):
    params3 = {
        "iterations": 5000,
        "learning_rate": 0.03,
        "depth": 7,
        "l2_leaf_reg": 3.0,
        "loss_function": "YetiRankPairwise",
        "custom_metric": ["NDCG:top=10", "MAP:top=10"],
        "eval_metric": "NDCG:top=10",
        "early_stopping_rounds": 200,
        "random_seed": 42,
        "task_type": "GPU",
        "devices": "0",
        "verbose": 200,
    }
    mlflow.log_params(params3)
    
    m3 = CatBoostRanker(**params3)
    m3.fit(train_pool, eval_set=test_pool)
    
    s3 = m3.get_best_score()["validation"]["NDCG:top=10;type=Base"]
    print(f"YetiRankPairwise NDCG@10: {s3:.4f} (iter {m3.get_best_iteration()})")
    mlflow.log_metric("best_ndcg_10", s3)
    m3.save_model("artifacts/catboost_v4_yrp.cbm")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
print(f"v2 baseline:       NDCG@10 = 0.3804")
print(f"v3 PairLogitPW:    NDCG@10 = 0.4121")
print(f"v4 PairLogitPW:    NDCG@10 = {s1:.4f}")
print(f"v4 YetiRank:       NDCG@10 = {s2:.4f}")
print(f"v4 YetiRankPW:     NDCG@10 = {s3:.4f}")

# Feature importance (лучшая модель)
scores = {'PairLogitPW': (s1, m1), 'YetiRank': (s2, m2), 'YetiRankPW': (s3, m3)}
best_name, (best_score, best_model) = max(scores.items(), key=lambda x: x[0])

fi = best_model.get_feature_importance(train_pool)
fi_df = pd.DataFrame({'feature': feature_cols, 'importance': fi}).sort_values('importance', ascending=False)
print(f"\nTop-40 features ({best_name}):")
print(fi_df.head(40).to_string(index=False))
fi_df.to_csv('artifacts/feature_importance_v4.csv', index=False)


GPU: 1
Features: 229


2026/05/15 22:24:45 INFO mlflow.tracking.fluent: Experiment with name 'Steam_RecSys_v4' does not exist. Creating a new experiment.



Run 1: PairLogitPairwise
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because MAP, NDCG is/are not implemented for GPU
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric MAP:top=10 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2470349	best: 0.2470349 (0)	total: 130ms	remaining: 10m 52s
200:	test: 0.3757735	best: 0.3761710 (198)	total: 13.8s	remaining: 5m 30s
400:	test: 0.3941596	best: 0.3943122 (397)	total: 27.4s	remaining: 5m 13s
600:	test: 0.4024520	best: 0.4025548 (592)	total: 40.9s	remaining: 4m 59s
800:	test: 0.4035810	best: 0.4046010 (758)	total: 54.5s	remaining: 4m 45s
bestTest = 0.40460104
bestIteration = 758
Shrink model to first 759 iterations.
PairLogitPW NDCG@10: 0.4046 (iter 758)

Run 2: YetiRank
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because MAP, NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric MAP:top=10 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2670973	best: 0.2670973 (0)	total: 59.1ms	remaining: 4m 55s
200:	test: 0.3472184	best: 0.3472184 (200)	total: 6.77s	remaining: 2m 41s
400:	test: 0.3724741	best: 0.3727001 (399)	total: 13.4s	remaining: 2m 34s
600:	test: 0.3810941	best: 0.3827429 (581)	total: 20.1s	remaining: 2m 26s
800:	test: 0.3867552	best: 0.3867552 (800)	total: 26.7s	remaining: 2m 19s
1000:	test: 0.3919810	best: 0.3919810 (1000)	total: 33.3s	remaining: 2m 13s
1200:	test: 0.3945925	best: 0.3946406 (1199)	total: 40s	remaining: 2m 6s
1400:	test: 0.3958511	best: 0.3968068 (1363)	total: 46.6s	remaining: 1m 59s
1600:	test: 0.3973107	best: 0.3974496 (1587)	total: 53.3s	remaining: 1m 53s
1800:	test: 0.3980928	best: 0.3986413 (1775)	total: 59.9s	remaining: 1m 46s
2000:	test: 0.4010145	best: 0.4010209 (1999)	total: 1m 6s	remaining: 1m 39s
2200:	test: 0.4022965	best: 0.4025760 (2182)	total: 1m 13s	remaining: 1m 33s
2400:	test: 0.4039601	best: 0.4040929 (2303)	total: 1m 19s	remaining: 1m 26s
2600:	test: 0.4056230	best

Default metric period is 5 because MAP, NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric MAP:top=10 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.1984970	best: 0.1984970 (0)	total: 306ms	remaining: 25m 30s
200:	test: 0.3783490	best: 0.3792136 (199)	total: 59.1s	remaining: 23m 31s
400:	test: 0.3954273	best: 0.3954352 (399)	total: 1m 57s	remaining: 22m 31s
600:	test: 0.4006174	best: 0.4006888 (583)	total: 2m 56s	remaining: 21m 33s
800:	test: 0.4049921	best: 0.4055146 (791)	total: 3m 55s	remaining: 20m 34s
1000:	test: 0.4105009	best: 0.4112556 (990)	total: 4m 54s	remaining: 19m 36s
1200:	test: 0.4124096	best: 0.4134625 (1100)	total: 5m 53s	remaining: 18m 37s
1400:	test: 0.4132808	best: 0.4145787 (1321)	total: 6m 52s	remaining: 17m 38s
1600:	test: 0.4156839	best: 0.4164643 (1547)	total: 7m 51s	remaining: 16m 40s
1800:	test: 0.4166709	best: 0.4175701 (1767)	total: 8m 50s	remaining: 15m 41s
2000:	test: 0.4182932	best: 0.4188440 (1939)	total: 9m 49s	remaining: 14m 43s
2200:	test: 0.4190107	best: 0.4199044 (2144)	total: 10m 48s	remaining: 13m 44s
2400:	test: 0.4223915	best: 0.4224336 (2398)	total: 11m 47s	remaining: 12m 45s
2